# Turkish Legal Retrieval Corpus Preparation

## Purpose

This notebook prepares a retrieval corpus from the merged legal QA dataset produced in notebook 01_dataset_preparation.ipynb.

**What this notebook does:**
- Loads the merged normalized legal QA dataset from Google Drive
- Creates structured retrieval documents combining question, answer, source, and category fields
- Applies text cleaning preserving Turkish characters
- Chunks documents by character length with overlap for better retrieval coverage
- Generates metadata (doc_id, chunk_id, text lengths, etc.)
- Performs quality checks on the corpus
- Saves retrieval corpus in CSV and JSONL formats

**What this notebook does NOT do:**
- Build embeddings or vector indices
- Implement BM25 or keyword search
- Evaluate retrieval performance
- Use sentence-transformers or deep learning models

**Note:** This is a first-pass corpus from QA data. A future step should augment with legal document corpus (laws, articles, precedents) for stronger retrieval grounding.

## 1. Environment Setup

In [ ]:
import os
import json
import uuid
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional, Tuple

## 2. Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✓ Google Drive mounted")

## 3. Configuration and Paths

In [ ]:
# Project configuration
PROJECT_ROOT = "/content/drive/My Drive/nlp-rag-project"
MERGED_DATA_PATH = f"{PROJECT_ROOT}/data/processed/merged_legal_qa.csv"
RETRIEVAL_OUTPUT_DIR = f"{PROJECT_ROOT}/data/retrieval"

# Chunking configuration
CHUNK_MAX_CHARS = 800  # Character limit per chunk
CHUNK_OVERLAP = 100    # Character overlap between chunks

# Create output directory
Path(RETRIEVAL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input data: {MERGED_DATA_PATH}")
print(f"Output directory: {RETRIEVAL_OUTPUT_DIR}")
print(f"Chunk settings: max_chars={CHUNK_MAX_CHARS}, overlap={CHUNK_OVERLAP}")

## 4. Load Merged Dataset

In [ ]:
# Load merged legal QA dataset
df_merged = pd.read_csv(MERGED_DATA_PATH)
print(f"✓ Loaded merged dataset: {df_merged.shape[0]} rows, {df_merged.shape[1]} columns")
print(f"\nColumns: {list(df_merged.columns)}")
print(f"\nData types:\n{df_merged.dtypes}")

## 5. Dataset Inspection

In [ ]:
print("="*80)
print("DATASET OVERVIEW")
print("="*80)

print(f"\nShape: {df_merged.shape}")
print(f"\nMissing values:")
print(df_merged.isnull().sum())

print(f"\nDataset composition:")
print(df_merged["dataset_name"].value_counts())

print(f"\nSample rows:")
print(df_merged.head(2))

In [ ]:
# Check source and category coverage
print(f"\nSource coverage:")
print(f"  - Non-null: {df_merged['source'].notna().sum()}")
print(f"  - Null: {df_merged['source'].isna().sum()}")
if df_merged['source'].notna().any():
    print(f"  - Unique sources: {df_merged['source'].nunique()}")

print(f"\nCategory coverage:")
print(f"  - Non-null: {df_merged['category'].notna().sum()}")
print(f"  - Null: {df_merged['category'].isna().sum()}")
if df_merged['category'].notna().any():
    print(f"  - Unique categories: {df_merged['category'].nunique()}")
    print(f"\n  Top categories:")
    print(df_merged['category'].value_counts().head(10))

## 6. Build Retrieval Documents

In [ ]:
def build_retrieval_text(row: pd.Series) -> str:
    """
    Build full retrieval text from row fields.
    Includes source, category, question, and answer when available.
    """
    parts = []
    
    if pd.notna(row.get('source')):
        parts.append(f"Source: {row['source']}")
    
    if pd.notna(row.get('category')):
        parts.append(f"Category: {row['category']}")
    
    if pd.notna(row.get('question')):
        parts.append(f"Question: {row['question']}")
    
    if pd.notna(row.get('answer')):
        parts.append(f"Answer: {row['answer']}")
    
    return "\n".join(parts)


def build_candidate_text(row: pd.Series) -> str:
    """
    Build shorter candidate text from question and answer.
    """
    parts = []
    
    if pd.notna(row.get('question')):
        parts.append(str(row['question']))
    
    if pd.notna(row.get('answer')):
        parts.append(str(row['answer']))
    
    return "\n".join(parts)


# Build retrieval text and candidate text for each row
df_merged['retrieval_text'] = df_merged.apply(build_retrieval_text, axis=1)
df_merged['candidate_text'] = df_merged.apply(build_candidate_text, axis=1)

print(f"✓ Built retrieval text fields")
print(f"\nExample retrieval text:\n")
print(df_merged['retrieval_text'].iloc[0][:300])

## 7. Text Cleaning for Retrieval

In [ ]:
def clean_retrieval_text(text: str) -> str:
    """
    Clean text for retrieval while preserving Turkish characters.
    """
    if not isinstance(text, str):
        return ""
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    # Collapse repeated spaces and newlines
    text = " ".join(text.split())
    
    return text


# Apply cleaning
df_merged['retrieval_text_clean'] = df_merged['retrieval_text'].apply(clean_retrieval_text)
df_merged['candidate_text_clean'] = df_merged['candidate_text'].apply(clean_retrieval_text)

print(f"✓ Cleaned retrieval text fields")
print(f"\nExample cleaned retrieval text:\n")
print(df_merged['retrieval_text_clean'].iloc[0][:300])

## 8. Text Chunking Strategy

In [ ]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100) -> List[str]:
    """
    Chunk text by character length with overlap.
    Preserves Turkish characters and avoids empty chunks.
    
    Args:
        text: Text to chunk
        max_chars: Maximum characters per chunk
        overlap: Character overlap between consecutive chunks
    
    Returns:
        List of chunks (strings)
    """
    if not isinstance(text, str) or len(text) == 0:
        return []
    
    # If text is shorter than max_chars, keep as single chunk
    if len(text) <= max_chars:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(text):
        # Calculate end position
        end = start + max_chars
        
        # If this is the last chunk, include remainder
        if end >= len(text):
            chunk = text[start:]
            if chunk.strip():  # Avoid empty chunks
                chunks.append(chunk)
            break
        
        # Try to break at a word boundary (space)
        # Look for last space within chunk
        last_space = text.rfind(' ', start, end)
        if last_space > start:
            end = last_space
        
        chunk = text[start:end].strip()
        if chunk:  # Avoid empty chunks
            chunks.append(chunk)
        
        # Move start position with overlap
        start = end - overlap if end > overlap else end
    
    return chunks


# Test chunking function
test_text = df_merged['retrieval_text_clean'].iloc[0]
test_chunks = chunk_text(test_text, max_chars=CHUNK_MAX_CHARS, overlap=CHUNK_OVERLAP)
print(f"✓ Chunking function ready")
print(f"\nExample: text length={len(test_text)}, chunks={len(test_chunks)}")
if test_chunks:
    print(f"  - Chunk 1 length: {len(test_chunks[0])}")
    if len(test_chunks) > 1:
        print(f"  - Chunk 2 length: {len(test_chunks[1])}")

## 9. Create Retrieval Corpus with Chunks

In [ ]:
# Generate corpus with chunks
corpus_rows = []

for original_idx, row in df_merged.iterrows():
    # Get text to chunk
    text_to_chunk = row['retrieval_text_clean']
    
    # Generate unique doc_id from UUID
    doc_id = f"doc_{original_idx:05d}_{str(uuid.uuid4())[:8]}"
    
    # Chunk the text
    chunks = chunk_text(text_to_chunk, max_chars=CHUNK_MAX_CHARS, overlap=CHUNK_OVERLAP)
    
    # Create corpus row for each chunk
    for chunk_idx, chunk_text_val in enumerate(chunks):
        chunk_id = f"{doc_id}_chunk_{chunk_idx}"
        
        corpus_rows.append({
            'doc_id': doc_id,
            'chunk_id': chunk_id,
            'chunk_index': chunk_idx,
            'original_row_index': original_idx,
            'question': row['question'],
            'answer': row['answer'],
            'source': row['source'],
            'category': row['category'],
            'dataset_name': row['dataset_name'],
            'split': row['split'],
            'quality_score': row['quality_score'],
            'full_text': text_to_chunk,
            'chunk_text': chunk_text_val,
            'text_length': len(text_to_chunk),
            'chunk_length': len(chunk_text_val),
            'n_chunks': len(chunks),
        })

# Create retrieval corpus DataFrame
df_corpus = pd.DataFrame(corpus_rows)
print(f"✓ Created retrieval corpus")
print(f"  - Original documents: {df_merged.shape[0]}")
print(f"  - Total chunks: {len(df_corpus)}")
print(f"  - Average chunks per document: {len(df_corpus) / df_merged.shape[0]:.2f}")

In [ ]:
# Show corpus sample
print(f"\nCorpus sample (first 3 rows):")
print(df_corpus[['doc_id', 'chunk_index', 'chunk_length', 'source', 'category']].head(3))

print(f"\nExample chunk text:")
print(df_corpus['chunk_text'].iloc[0][:200])

## 10. Corpus Quality Checks

In [ ]:
print("="*80)
print("RETRIEVAL CORPUS QUALITY REPORT")
print("="*80)

print(f"\nCorpus Statistics:")
print(f"  - Total chunks: {len(df_corpus)}")
print(f"  - Original documents: {df_merged.shape[0]}")
print(f"  - Avg chunks/document: {len(df_corpus) / df_merged.shape[0]:.2f}")

print(f"\nChunk Length Statistics:")
print(f"  - Min: {df_corpus['chunk_length'].min()}")
print(f"  - Max: {df_corpus['chunk_length'].max()}")
print(f"  - Mean: {df_corpus['chunk_length'].mean():.0f}")
print(f"  - Median: {df_corpus['chunk_length'].median():.0f}")
print(f"  - Std: {df_corpus['chunk_length'].std():.0f}")

print(f"\nFull Text Length Statistics:")
print(f"  - Min: {df_corpus['text_length'].min()}")
print(f"  - Max: {df_corpus['text_length'].max()}")
print(f"  - Mean: {df_corpus['text_length'].mean():.0f}")
print(f"  - Median: {df_corpus['text_length'].median():.0f}")

In [ ]:
print(f"\nMetadata Coverage:")
print(f"  - Source present: {df_corpus['source'].notna().sum()} / {len(df_corpus)}")
print(f"  - Category present: {df_corpus['category'].notna().sum()} / {len(df_corpus)}")
print(f"  - Quality score present: {df_corpus['quality_score'].notna().sum()} / {len(df_corpus)}")

print(f"\nTop Sources:")
top_sources = df_corpus[df_corpus['source'].notna()]['source'].value_counts().head(5)
if len(top_sources) > 0:
    for source, count in top_sources.items():
        print(f"  - {source}: {count}")
else:
    print("  (No sources available)")

print(f"\nTop Categories:")
top_cats = df_corpus[df_corpus['category'].notna()]['category'].value_counts().head(5)
if len(top_cats) > 0:
    for cat, count in top_cats.items():
        print(f"  - {cat}: {count}")
else:
    print("  (No categories available)")

print(f"\nDataset composition:")
print(df_corpus['dataset_name'].value_counts())

In [ ]:
# Show sample chunks
print("\n" + "="*80)
print("SAMPLE CHUNKS FROM CORPUS")
print("="*80)

for idx in range(min(3, len(df_corpus))):
    row = df_corpus.iloc[idx]
    print(f"\n--- Chunk {idx + 1} ---")
    print(f"doc_id: {row['doc_id']}")
    print(f"chunk_index: {row['chunk_index']} / {row['n_chunks']}")
    print(f"source: {row['source']}")
    print(f"category: {row['category']}")
    print(f"chunk_length: {row['chunk_length']}")
    print(f"\nChunk text:\n{row['chunk_text'][:250]}...")

## 11. Save Retrieval Corpus

In [ ]:
# Reorder columns for clarity
column_order = [
    'doc_id', 'chunk_id', 'chunk_index',
    'original_row_index',
    'question', 'answer',
    'source', 'category',
    'dataset_name', 'split', 'quality_score',
    'full_text', 'chunk_text',
    'text_length', 'chunk_length', 'n_chunks'
]

df_corpus = df_corpus[column_order]

print(f"✓ Finalized corpus DataFrame")
print(f"  Shape: {df_corpus.shape}")
print(f"  Columns: {list(df_corpus.columns)}")

In [ ]:
# Save full corpus as CSV
csv_output = f"{RETRIEVAL_OUTPUT_DIR}/retrieval_corpus_full.csv"
df_corpus.to_csv(csv_output, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {csv_output}")
print(f"  Rows: {len(df_corpus)}")
print(f"  Size: {Path(csv_output).stat().st_size / (1024*1024):.2f} MB")

In [ ]:
# Save full corpus as JSONL
jsonl_output = f"{RETRIEVAL_OUTPUT_DIR}/retrieval_corpus_full.jsonl"

with open(jsonl_output, 'w', encoding='utf-8') as f:
    for idx, row in df_corpus.iterrows():
        json_record = row.to_dict()
        # Convert NaN to None for JSON serialization
        json_record = {
            k: (None if pd.isna(v) else v) for k, v in json_record.items()
        }
        f.write(json.dumps(json_record, ensure_ascii=False) + "\n")

print(f"✓ Saved: {jsonl_output}")
print(f"  Records: {len(df_corpus)}")
print(f"  Size: {Path(jsonl_output).stat().st_size / (1024*1024):.2f} MB")

In [ ]:
# Save preview (first 200 rows)
preview_output = f"{RETRIEVAL_OUTPUT_DIR}/retrieval_corpus_preview.csv"
df_corpus.head(200).to_csv(preview_output, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {preview_output}")
print(f"  Preview rows: {min(200, len(df_corpus))}")

## 12. Summary and Next Steps

In [ ]:
print("\n" + "="*80)
print("RETRIEVAL CORPUS PREPARATION COMPLETE")
print("="*80)

print(f"\nCorpus Summary:")
print(f"  • Original documents: {df_merged.shape[0]:,}")
print(f"  • Total chunks: {len(df_corpus):,}")
print(f"  • Avg chunks per document: {len(df_corpus) / df_merged.shape[0]:.2f}")
print(f"  • Avg chunk length: {df_corpus['chunk_length'].mean():.0f} characters")

print(f"\nOutput Files (saved to {RETRIEVAL_OUTPUT_DIR}):")
output_files = [
    ("retrieval_corpus_full.csv", "Complete corpus with all chunks"),
    ("retrieval_corpus_full.jsonl", "Same data in JSONL format"),
    ("retrieval_corpus_preview.csv", "First 200 rows for preview"),
]
for fname, desc in output_files:
    print(f"  ✓ {fname}")
    print(f"    {desc}")

print(f"\nImportant Notes:")
print(f"  • This is a FIRST-PASS retrieval corpus from QA data")
print(f"  • Ideal corpus would include separate legal documents (laws, articles, precedents)")
print(f"  • Current corpus is suitable for baseline retrieval experiments")
print(f"  • Future steps should augment with a law/article-based document corpus")

print(f"\nNext Steps:")
print(f"  1. ✓ Dataset preparation (notebook 01)")
print(f"  2. ✓ Retrieval corpus preparation (this notebook)")
print(f"  3. → Extract and prepare legal document corpus (laws, articles)")
print(f"  4. → Build embeddings and create FAISS index")
print(f"  5. → Implement BM25 ranking component")
print(f"  6. → Create retrieval evaluation benchmark")
print(f"  7. → Integrate LLM answer generation")
print(f"  8. → Evaluate full RAG pipeline")